# 🏙️ Model 01 — City Day AQI Prediction
**AirVintage Production ML Pipeline — Deployment-Ready**

| Property | Value |
|----------|-------|
| Dataset | `city_day_cleaned.csv` |
| Algorithm | **XGBoost Regressor + Optuna** |
| Target | `AQI` |

## ✅ Production Input Contract
Only features available from **free public APIs**:

| Input | Source | Free? |
|-------|--------|-------|
| `PM2.5`, `PM10` | OpenAQ / CPCB API | ✅ |
| `NO2`, `SO2`, `CO`, `O3` | OpenAQ / CPCB API | ✅ |
| `City name` | User input / GPS | ✅ |
| `Date/Time` | System clock | ✅ |

**Dropped from training:** `NO`, `NOx`, `NH3`, `Benzene`, `Toluene`, `Xylene`  
*(industrial VOC sensors — not available in free APIs)*

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

from airvintage_ml import (
    aqi_to_bucket, health_advisory,
    add_temporal_features, add_pollutant_interactions,
    encode_categorical, fillna_production, add_entity_stats,
    compute_metrics, print_metrics_table,
    plot_actual_vs_pred, plot_residuals,
    plot_feature_importance, plot_learning_curve_xgb,
    update_model_registry,
)

DATASET    = 'city_day_cleaned.csv'
MODEL_KEY  = 'city_day'
MODELS_DIR = 'models'
REGISTRY   = f'{MODELS_DIR}/model_registry.json'
SEED       = 42
os.makedirs(MODELS_DIR, exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── PRODUCTION FEATURE CONTRACT ─────────────────────────────
# Only pollutants available from free APIs (OpenAQ / CPCB)
CORE_POLLUTANTS = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']
# Dropped: NO, NOx, NH3, Benzene, Toluene, Xylene (not freely available)

print(f'XGBoost {xgb.__version__} | Optuna {optuna.__version__}')
print(f'Production pollutants: {CORE_POLLUTANTS}')

In [ ]:
# ── Load
df = pd.read_csv(DATASET, parse_dates=['Date'])
print(f'Shape  : {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Cities : {df["City"].nunique()}')
print(f'All columns: {list(df.columns)}')

In [ ]:
# ── EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(df['AQI'].dropna(), bins=60, color='#667eea', edgecolor='white', alpha=0.85)
axes[0,0].set_title('AQI Distribution', fontweight='bold')

bc = df['AQI_Bucket'].value_counts()
axes[0,1].pie(bc.values, labels=bc.index, autopct='%1.1f%%',
              colors=['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad','#2c3e50'][:len(bc)])
axes[0,1].set_title('AQI Category Split', fontweight='bold')

# Correlation of CORE pollutants with AQI
avail = [c for c in CORE_POLLUTANTS if c in df.columns]
corr = df[avail + ['AQI']].corr()['AQI'].drop('AQI').abs().sort_values()
axes[0,2].barh(corr.index, corr.values, color='#4facfe')
axes[0,2].set_title('Core Pollutant Correlation with AQI', fontweight='bold')

top = df.groupby('City')['AQI'].mean().sort_values(ascending=False).head(10)
axes[1,0].barh(top.index[::-1], top.values[::-1], color=plt.cm.Reds(np.linspace(0.4,0.9,10)))
axes[1,0].set_title('Top 10 Polluted Cities', fontweight='bold')

df['Month_tmp'] = df['Date'].dt.month
monthly = df.groupby('Month_tmp')['AQI'].mean()
axes[1,1].plot(range(1,13), monthly.values, 'o-', color='#667eea', lw=2.5)
axes[1,1].set_xticks(range(1,13))
axes[1,1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1,1].set_title('Monthly Avg AQI', fontweight='bold')

# Core pollutants boxplot
df[[c for c in CORE_POLLUTANTS if c in df.columns]].boxplot(ax=axes[1,2])
axes[1,2].set_title('Core Pollutant Distributions', fontweight='bold')
axes[1,2].tick_params(axis='x', rotation=30)

fig.suptitle('City Day EDA — Production Feature Set', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/eda_01_city_day.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  FEATURE ENGINEERING — PRODUCTION ONLY                   ║
# ║  All features derivable at inference time from free APIs ║
# ╚══════════════════════════════════════════════════════════╝
df_feat = df.copy()

# 1. Keep only production-available pollutants
avail_pollutants = [c for c in CORE_POLLUTANTS if c in df_feat.columns]

# 2. Temporal features — derived from datetime (free, always available)
df_feat = add_temporal_features(df_feat, 'Date')

# 3. Pollutant interactions — computed from core pollutants (no extra API)
df_feat['PM_ratio']  = df_feat['PM2.5'] / (df_feat['PM10'] + 1e-6)
df_feat['Total_PM']  = df_feat['PM2.5'] + df_feat['PM10']
df_feat['NO2_SO2']   = df_feat['NO2']   + df_feat['SO2']
df_feat['CO_O3']     = df_feat['CO']    * df_feat['O3']   # photochemical proxy

# 4. City encoding — derived from location (free, user provides city name)
df_feat, le_city = encode_categorical(df_feat, 'City')
df_feat, city_stats = add_entity_stats(df_feat, 'City_encoded', 'AQI')

# ── Define exact production feature columns
DROP = ['City', 'Date', 'AQI', 'AQI_Bucket', 'Month_tmp',
        # Drop raw non-API pollutants even if present in training dataset
        'NO', 'NOx', 'NH3', 'Benzene', 'Toluene', 'Xylene']
FEATURE_COLS = [c for c in df_feat.columns if c not in DROP]

df_clean = df_feat.dropna(subset=['AQI']).copy()
df_clean = fillna_production(df_clean)
X = df_clean[FEATURE_COLS]
y = df_clean['AQI']

print(f'Production features ({len(FEATURE_COLS)}):')
for i, f in enumerate(FEATURE_COLS): print(f'   {i+1:2d}. {f}')
print(f'\nSamples: {X.shape[0]:,}  |  AQI: {y.min():.0f}–{y.max():.0f}')

In [ ]:
# ── Split
X_train,X_tmp,y_train,y_tmp = train_test_split(X,y,test_size=0.30,random_state=SEED)
X_val,X_test,y_val,y_test   = train_test_split(X_tmp,y_tmp,test_size=0.50,random_state=SEED)
print(f'Train:{X_train.shape[0]:,} | Val:{X_val.shape[0]:,} | Test:{X_test.shape[0]:,}')

In [ ]:
# ── Optuna Tuning
def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 4, 10),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'tree_method':'hist', 'random_state':SEED, 'verbosity':0,
    }
    m = xgb.XGBRegressor(**params, early_stopping_rounds=30, eval_metric='rmse')
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return float(np.sqrt(np.mean((y_val - m.predict(X_val))**2)))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)
best_params = study.best_params
print(f'Best Val RMSE: {study.best_value:.4f}')

In [ ]:
# ── Final Model
model = xgb.XGBRegressor(
    **best_params, tree_method='hist', random_state=SEED,
    early_stopping_rounds=50, eval_metric='rmse', verbosity=0
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

# 5-Fold CV
cv_params = dict(best_params, n_estimators=model.best_iteration or 500)
cv = xgb.XGBRegressor(**cv_params, tree_method='hist', random_state=SEED, verbosity=0)
kr2 = cross_val_score(cv, X, y, cv=KFold(5,shuffle=True,random_state=SEED), scoring='r2', n_jobs=-1)
print(f'5-Fold CV R2: {kr2.mean():.4f} +/- {kr2.std():.4f}')

In [ ]:
# ── Evaluate
train_m = compute_metrics(y_train, model.predict(X_train))
val_m   = compute_metrics(y_val,   model.predict(X_val))
test_m  = compute_metrics(y_test,  model.predict(X_test))
print_metrics_table({'Train': train_m, 'Val': val_m, 'Test': test_m})

plot_learning_curve_xgb(model.evals_result(), model.best_iteration,
    'XGBoost — City Day', f'{MODELS_DIR}/lc_01_city_day.png')
plot_actual_vs_pred(y_test, model.predict(X_test),
    'Actual vs Predicted — City Day', f'{MODELS_DIR}/avp_01_city_day.png')
plot_residuals(y_test, model.predict(X_test),
    'Residuals — City Day', f'{MODELS_DIR}/res_01_city_day.png')
plot_feature_importance(FEATURE_COLS, model.feature_importances_,
    'Feature Importance — City Day', savepath=f'{MODELS_DIR}/fi_01_city_day.png')

In [ ]:
# ── Save
model.save_model(f'{MODELS_DIR}/01_city_day_xgb.json')
joblib.dump(le_city, f'{MODELS_DIR}/01_city_day_le_city.pkl')
city_stats.to_csv(f'{MODELS_DIR}/01_city_day_city_stats.csv', index=False)
with open(f'{MODELS_DIR}/01_city_day_features.json','w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
# Save production input schema
with open(f'{MODELS_DIR}/01_city_day_input_schema.json','w') as f:
    json.dump({'required_pollutants': CORE_POLLUTANTS,
               'required_inputs': ['city', 'date'] + CORE_POLLUTANTS,
               'note': 'All pollutants from OpenAQ/CPCB free API'}, f, indent=2)

update_model_registry(REGISTRY, MODEL_KEY, 'XGBoost + Optuna', DATASET, test_m,
    model_paths={
        'model'       : f'{MODELS_DIR}/01_city_day_xgb.json',
        'encoder'     : f'{MODELS_DIR}/01_city_day_le_city.pkl',
        'city_stats'  : f'{MODELS_DIR}/01_city_day_city_stats.csv',
        'features'    : f'{MODELS_DIR}/01_city_day_features.json',
        'input_schema': f'{MODELS_DIR}/01_city_day_input_schema.json',
    }, feature_count=len(FEATURE_COLS),
    notes='Inputs: PM2.5,PM10,NO2,SO2,CO,O3 + city + date (all free API)')
print('Model 01 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  PRODUCTION INFERENCE FUNCTION                           ║
# ║  Minimal inputs — everything from free APIs              ║
# ╚══════════════════════════════════════════════════════════╝
def predict_city_day(
    city: str,
    date: str,
    pm25: float, pm10: float,
    no2: float, so2: float, co: float, o3: float,
) -> dict:
    """
    Predict daily AQI for a city.

    Parameters (all from free APIs)
    --------------------------------
    city       : City name          (user input / GPS reverse geocode)
    date       : 'YYYY-MM-DD'       (system clock)
    pm25, pm10 : ug/m3              (OpenAQ / CPCB API)
    no2, so2   : ug/m3              (OpenAQ / CPCB API)
    co         : mg/m3              (OpenAQ / CPCB API)
    o3         : ug/m3              (OpenAQ / CPCB API)

    Returns
    -------
    dict: predicted_aqi, aqi_category, health_advisory
    """
    row = pd.DataFrame([{'City': city, 'Date': date,
                          'PM2.5': pm25, 'PM10': pm10,
                          'NO2': no2, 'SO2': so2, 'CO': co, 'O3': o3}])
    row['Date'] = pd.to_datetime(row['Date'])
    row = add_temporal_features(row, 'Date')

    row['PM_ratio'] = pm25 / (pm10 + 1e-6)
    row['Total_PM'] = pm25 + pm10
    row['NO2_SO2']  = no2  + so2
    row['CO_O3']    = co   * o3

    c_enc = int(le_city.transform([city])[0]) if city in le_city.classes_ else -1
    row['City_encoded'] = c_enc
    sr = city_stats[city_stats['City_encoded'] == c_enc]
    for col in [c for c in city_stats.columns if 'AQI' in c]:
        row[col] = float(sr[col].values[0]) if len(sr) > 0 else 150.0

    X_in = row[FEATURE_COLS].fillna(0)
    aqi  = float(model.predict(X_in)[0])
    b    = aqi_to_bucket(aqi)
    return {'model':'city_day','city':city,'date':date,
            'predicted_aqi':round(aqi,2),'aqi_category':b,
            'health_advisory':health_advisory(b)}


# Demo — exactly what AirVintage backend will call
result = predict_city_day(
    city='Delhi', date='2025-03-15',
    pm25=85.0, pm10=120.0,
    no2=40.0, so2=10.0, co=1.2, o3=30.0
)
print('Prediction result:')
for k, v in result.items(): print(f'  {k:20s}: {v}')

---
## Summary — Production API Contract

**To predict AQI for a city, your backend needs only:**
```json
{
  "city": "Delhi",
  "date": "2025-03-15",
  "pm25": 85.0,
  "pm10": 120.0,
  "no2":  40.0,
  "so2":  10.0,
  "co":   1.2,
  "o3":   30.0
}
```
**Data sources:** OpenAQ or CPCB open API (both free)

Saved at: `models/01_city_day_xgb.json`
